# Lecture 2: Building a Neural Network — Plain Language Guide

This notebook explains every concept from the neural network exercise in simple, everyday language — no maths degree required.

**The business scenario throughout this notebook:**
A retail company wants to predict which website visitors are likely to make a purchase — so they can personalise offers in real time and increase conversion rates.

---

**What we cover:**
1. What a neural network actually is
2. The dataset — what we are working with
3. Preparing data for AI
4. Turning data into the format the model needs
5. Building the neural network
6. The learning mechanism — loss and optimiser
7. The training loop — how the model actually learns
8. Reading the results — loss curves
9. Evaluating performance — beyond simple accuracy


## Section 1: What Is a Neural Network?

---

### The One-Sentence Version

A neural network is a system that learns to make predictions by repeatedly adjusting thousands of internal dials — until its predictions are as accurate as possible.

---

### The Business Analogy

Imagine you are training a new analyst to predict which customers will buy. On day one, the analyst knows nothing, so they make random guesses. After each prediction, you tell them whether they were right or wrong, and they adjust their thinking slightly.

After seeing thousands of examples, the analyst has developed an internal sense of which patterns lead to a purchase — even for customers they have never seen before. They cannot fully explain their reasoning, but their predictions are accurate.

A neural network works the same way. It starts with random guesses, gets feedback on how wrong it was, adjusts its internal settings, and repeats — millions of times.

---

### How It Is Structured

A neural network is organised in layers:

| Layer | Role | Business analogy |
|-------|------|-----------------|
| **Input layer** | Receives the raw data | The raw customer data arriving on your desk |
| **Hidden layers** | Find patterns and relationships in the data | The analyst's internal reasoning process |
| **Output layer** | Produces the final prediction | The analyst's final answer: buy or not buy |

The "hidden" layers are where the learning happens. You cannot directly see what they are doing — but they are building increasingly abstract representations of the patterns in your data.

---

### What Makes It "Deep"?

A neural network with multiple hidden layers is called a **deep neural network** — hence "deep learning." More layers allow the network to learn more complex patterns. The exercise in this lecture uses two hidden layers, which is sufficient for most tabular business prediction tasks.


## Section 2: The Dataset — Online Shoppers Purchasing Intention

---

### What Are We Trying to Predict?

The goal is **binary classification**: for each website session, predict whether it ended in a purchase (`True`) or not (`False`). This is a yes/no question — the most common type of business prediction problem.

Examples of similar business questions:
- Will this loan applicant default? (yes/no)
- Will this customer churn next month? (yes/no)
- Is this transaction fraudulent? (yes/no)

---

### What Data Do We Have?

The dataset contains 12,330 rows — one per website session — collected from a real e-commerce site. For each session, we know:

| Category | What it captures |
|----------|-----------------|
| Page visits | How many administrative, informational, and product pages the user visited |
| Time on pages | How long they spent on each type of page |
| Google Analytics signals | Bounce rate, exit rate, page value, proximity to special days |
| Session context | Month, operating system, browser, region, traffic source |
| Visitor profile | New visitor, returning visitor, or other |
| Timing | Whether the session occurred on a weekend |

**Target:** Did this session result in a purchase?

---

### The Imbalance Problem

Roughly **84% of sessions do not result in a purchase**. Only 16% do.

This is called a **class imbalance**, and it is extremely common in business datasets:
- Most loan applications are not fraudulent
- Most customers do not churn
- Most website visitors do not buy

This matters because a model that simply predicts "no purchase" every single time would be 84% accurate — but completely useless. We need metrics that reveal whether the model is genuinely learning to identify buyers, not just defaulting to the majority class.


## Section 3: Preparing Data — Why Raw Data Cannot Go Straight Into a Model

---

### The Core Problem

Neural networks only understand numbers. But real-world data contains:
- Text categories (e.g. "November", "Returning Visitor")
- Boolean flags (True/False)
- Numbers on wildly different scales (e.g. page visit count: 0–100 vs. time on page: 0–10,000 seconds)

Before training can begin, all of this must be standardised.

---

### Step 1: Encoding Categories — Turning Words Into Numbers

The dataset has columns like `Month` ("Jan", "Feb", …, "Dec") and `VisitorType` ("Returning Visitor", "New Visitor"). These cannot be fed to a neural network as text.

The solution is **one-hot encoding**: each possible category becomes its own column, with a 1 if that category applies and 0 otherwise.

**Example — Month column:**

| Original | Jan | Feb | Mar | … |
|----------|-----|-----|-----|---|
| "Jan"    | 1   | 0   | 0   | … |
| "Mar"    | 0   | 0   | 1   | … |

> **Why not just assign numbers?** Encoding "Jan=1, Feb=2, Mar=3" implies March is three times January, which is meaningless. One-hot encoding treats each category independently.

---

### Step 2: Splitting Into Training and Test Sets

The data is split into two parts:

- **Training set (80%)** — the model learns from this data
- **Test set (20%)** — the model is evaluated on data it has never seen

The test set acts as the final exam. A model that scores well only on training data has simply memorised the answers — like a student who memorises past exam papers but cannot handle new questions. This is called **overfitting**.

The split uses **stratification** — ensuring both sets have the same proportion of buyers and non-buyers. Without this, random chance could put most of the rare purchase cases into one set, making evaluation misleading.

---

### Step 3: Normalisation — Putting All Features on the Same Scale

Some features have values in the range 0–1. Others are in the range 0–50,000. If left as-is, features with large values would dominate the model's learning simply due to their scale — not because they are more important.

Normalisation rescales all features so they have a mean of 0 and a standard deviation of 1. Every feature is now on a level playing field.

> **Critical rule:** The normalisation formula (mean and standard deviation) is calculated using **training data only** and then applied to the test set. Using test data to compute the formula would be like showing a student the exam answers before marking them — it artificially inflates performance.


## Section 4: DataLoaders — Feeding Data to the Model Efficiently

---

### Why Not Just Pass All the Data at Once?

With 12,330 rows, passing everything to the model simultaneously would require enormous memory — and would actually produce a less stable model.

Instead, the data is divided into **batches** of 64 rows at a time. The model processes one batch, updates its internal settings, processes the next batch, updates again, and so on.

---

### One Pass Through All Batches = One Epoch

When the model has processed every batch in the training set once, that is called one **epoch**. In this exercise, we train for 30 epochs — meaning the model sees the entire dataset 30 times in total.

> **Analogy:** Imagine revising for an exam. Reading your notes once is one pass. Reading them 30 times — each time reinforcing and refining your understanding — is 30 epochs. Each pass improves retention.

---

### Training vs. Test DataLoader

Two separate data pipelines are created:

- **Training DataLoader** — shuffles the data each epoch (prevents the model from memorising the order)
- **Test DataLoader** — no shuffling (order does not matter for evaluation, and consistency makes debugging easier)

This separation ensures the model is always evaluated on data it has not trained on — the integrity of the evaluation depends on this.


## Section 5: Building the Neural Network — What Is Actually Inside?

---

### Think of It Like a Pipeline of Decision-Makers

Imagine a customer walks into a shop. Three employees stand in a row:

1. **Employee 1** looks at the customer and spots broad patterns — "They came in on a weekday, they spent a long time on product pages."
2. **Employee 2** takes those observations and refines them — "Long time on product pages + low bounce rate = probably serious buyer."
3. **Employee 3** makes the final call — "70% chance this person will buy."

Each employee in this analogy is a **layer** in the neural network. They each pass their summary to the next layer, which builds on it.

---

### The Layers in This Exercise

Our network has four layers:

| Layer | Size | What it does |
|-------|------|--------------|
| Input | 1 neuron per feature (varies) | Receives the raw numbers for one session |
| Hidden Layer 1 | 64 neurons | Learns broad patterns from the data |
| Hidden Layer 2 | 32 neurons | Refines those patterns into more specific signals |
| Output | 1 neuron | Outputs a number between 0 and 1 — the purchase probability |

The number of neurons (64, 32) represents how much "thinking capacity" each layer has. More neurons = more capacity, but also more risk of memorising noise.

---

### What Are "Neurons" Exactly?

A neuron is simply a number — a score — that gets calculated from the layer before it. Each neuron takes all the values from the previous layer, multiplies each by a **weight** (how important is this input?), adds them up, and applies a small mathematical function.

> **Simple version:** Each neuron asks "given everything I've been told, how much should I fire?" The answer gets passed to the next layer.

---

### The Activation Functions

After each hidden layer, a function called **ReLU** is applied. ReLU simply says: "If the value is negative, make it zero. If it's positive, keep it."

This sounds trivial, but it is what allows the network to learn non-linear patterns — real-world relationships that are not simply "more X always means more Y."

At the output, a **Sigmoid** function squashes the final number into a value between 0 and 1. This makes the output interpretable as a probability: 0.8 means "80% chance of purchase."


## Section 6: The Loss Function and Optimiser — How the Model Knows It Is Wrong

---

### The Model Starts Knowing Nothing

When a neural network is first created, all its internal weights are set to small random numbers. This means its first predictions are essentially random guesses — some will be right by chance, most will be wrong.

The key question is: **how does it get better?**

---

### Step 1: Measuring the Mistake — The Loss Function

After each prediction, we compare what the model said to the correct answer. The **loss function** calculates a single number — the **loss** — that represents how wrong the model was.

Think of it like a score on a practice test:
- A low loss means the model's predictions were close to the truth — good job
- A high loss means the predictions were far off — needs improvement

In this exercise, the loss function used is called **Binary Cross-Entropy**. All you need to know is: it measures the gap between the model's predicted probability and the actual outcome (0 or 1). The bigger the gap, the higher the loss.

---

### Step 2: Fixing the Mistake — The Optimiser

Once the loss is calculated, the network needs to adjust its weights to reduce it. This is the job of the **optimiser**.

> **Analogy:** Imagine you are tuning thousands of volume dials on a mixing desk. You want the output to sound perfect. After each attempt, an assistant tells you how far off you are (the loss). You then nudge each dial slightly — turning up the ones that help, turning down the ones that hurt. Over thousands of attempts, the mix gets better and better.

The optimiser used here is called **Adam**. It is smart: it adjusts each weight by a different amount depending on how much that weight has been changing recently. This makes learning faster and more stable than simply adjusting everything by the same amount.

The **learning rate** (set to 0.001) controls how big each adjustment is:
- Too large: the model overshoots and never settles
- Too small: the model learns, but very slowly
- 0.001 is a safe, well-tested default for most problems


## Section 7: The Training Loop — How the Model Actually Learns

---

### Learning Happens in Cycles

Training a neural network is not a one-time event. It is a repeated cycle — run a prediction, measure how wrong it was, adjust the weights, repeat.

Here is what happens in each cycle (called an **epoch**):

---

### One Epoch, Step by Step

**1. Pick a batch of 64 examples from the training data**

Rather than learning from one example at a time (too slow) or all 9,000+ at once (too much memory), we process 64 rows at a time.

**2. Make predictions**

The network runs each of the 64 examples through all its layers and produces 64 predicted probabilities.

**3. Calculate the loss**

We compare the 64 predictions to the 64 correct answers. The loss function produces a single number that summarises how wrong the batch was.

**4. Work out which weights caused the mistake (backpropagation)**

This is the clever part. The network traces backwards through its layers to figure out which weights contributed most to the error. Think of it as the model asking: "which of my dials made things worse, and by how much?"

**5. Update the weights**

The optimiser nudges each weight slightly in the direction that reduces the loss.

**6. Repeat for every batch, then start the next epoch**

One epoch = the model has seen every training example once. After 30 epochs, it has seen all the data 30 times, each time becoming slightly better.

---

### Why 30 Epochs?

There is no universal right answer. Too few epochs and the model has not learned enough. Too many and it starts memorising the training data instead of learning general patterns (overfitting). 30 is a reasonable starting point for a dataset of this size.


## Section 8: The Loss Curve — Is the Model Actually Learning?

---

### What Is a Loss Curve?

After training, we can plot how the loss changed over each of the 30 epochs. This chart is called a **loss curve**, and it is one of the most useful tools for understanding whether training went well.

---

### What a Healthy Loss Curve Looks Like

- **Both lines start high** — the model is making poor predictions at the beginning (as expected)
- **Both lines decrease over time** — the model is learning and improving
- **Both lines level off near the end** — the model has reached its best performance
- **The two lines stay close together** — the model is generalising well, not just memorising

---

### Warning Sign: Overfitting

If the training loss keeps dropping but the test loss stops improving — or starts rising — this is called **overfitting**.

> **Analogy:** A student memorises every past exam paper word-for-word. They score 100% when tested on old papers, but fail the real exam because the questions are slightly different.

An overfitting model has memorised the training data rather than learning the underlying patterns. It performs well on data it has seen, but poorly on new data — which defeats the purpose of building the model.

---

### What to Look for in the Chart

| What you see | What it means |
|---|---|
| Both lines declining together | Model is learning well |
| Training loss low, test loss also low | Model generalises — this is the goal |
| Training loss low, test loss high | Overfitting — model memorised the training data |
| Both losses staying high | Underfitting — model has not learned enough |


## Section 9: Evaluating the Model — Why "84% Accurate" Can Be Completely Meaningless

---

### The Trap of Accuracy

After training, we want to know: how good is our model?

The first instinct is to look at **accuracy** — the percentage of predictions that were correct. But for this dataset, accuracy is misleading.

Recall: 84% of sessions in our data did not result in a purchase. So a model that says "no purchase" for every single visitor, without doing any learning at all, would score **84% accuracy**. That is not a good model — it is a useless one that has simply noticed which answer comes up most often.

---

### Better Metrics for Imbalanced Data

We need metrics that specifically measure how well the model identifies the rare positive case — the actual buyers.

**Precision — when the model says "purchase", how often is it right?**

> Imagine the model flags 100 visitors as likely buyers. Precision tells you how many of those 100 actually bought. High precision means fewer wasted marketing efforts.

**Recall — of all actual buyers, how many did the model catch?**

> Imagine there were 200 real buyers in the test set. Recall tells you how many of those 200 the model successfully identified. High recall means fewer missed opportunities.

**The trade-off:**
It is usually impossible to maximise both at once. Catching more buyers (high recall) often means flagging more non-buyers too (lower precision). The right balance depends on the business cost of each type of mistake.

**F1-score — the balance between precision and recall**

> The F1-score combines precision and recall into a single number. It is a useful summary when you care about both — a high F1 means you are catching a good proportion of real buyers without too many false alarms.

---

### Putting It Together

| Metric | Business meaning | Good when... |
|--------|-----------------|--------------|
| Accuracy | Overall % correct | Classes are balanced |
| Precision | Of predicted buyers, how many really bought | False positives are costly (e.g. expensive offers) |
| Recall | Of all real buyers, how many did we catch | Missing buyers is costly (e.g. lost revenue) |
| F1-score | Balance of precision and recall | Both types of mistakes matter |

For a purchase prediction model, **recall** is often the priority — missing a real buyer (a false negative) means lost revenue. But sending expensive personalised offers to non-buyers (false positives) also has a cost. F1-score helps navigate that balance.

---

### The Confusion Matrix

The confusion matrix is a simple 2×2 table that shows all four possible outcomes at once:

|  | Predicted: No Purchase | Predicted: Purchase |
|--|---|---|
| **Actual: No Purchase** | True Negative ✅ | False Positive ❌ |
| **Actual: Purchase** | False Negative ❌ | True Positive ✅ |

- **True Negative:** Correctly predicted "won't buy" — fine, no action needed
- **True Positive:** Correctly predicted "will buy" — great, we can target them
- **False Positive:** Predicted "will buy" but they didn't — wasted effort
- **False Negative:** Predicted "won't buy" but they did — missed opportunity


---

## Summary — What We Did in This Lecture

| Step | What happened | Why it matters |
|------|--------------|----------------|
| **1. Got the data** | Downloaded 12,330 real e-commerce sessions | Real data is messy — we need a starting point |
| **2. Explored the data** | Checked columns, types, missing values, class balance | Understand what you're working with before touching it |
| **3. Encoded categories** | Turned words like "November" into columns of 0s and 1s | Neural networks only understand numbers |
| **4. Split the data** | 80% for training, 20% for testing | The test set is the "final exam" the model has never seen |
| **5. Normalised the data** | Rescaled all features to the same range | Prevents large-valued features from drowning out small ones |
| **6. Created DataLoaders** | Divided data into batches of 64 | Efficient processing — the model updates weights after each batch |
| **7. Built the network** | 4 layers: input → 64 neurons → 32 neurons → 1 output | More layers = more capacity to learn complex patterns |
| **8. Set the loss function** | Binary Cross-Entropy | Measures how wrong each prediction is on a 0–1 scale |
| **9. Set the optimiser** | Adam with learning rate 0.001 | Adjusts each weight intelligently to reduce the loss |
| **10. Trained for 30 epochs** | The model saw all data 30 times | Each pass improves accuracy by adjusting thousands of weights |
| **11. Read the loss curve** | Checked both training and test loss declined | Confirms the model is learning, not just memorising |
| **12. Evaluated properly** | Used precision, recall, F1, confusion matrix | Accuracy alone is misleading when classes are imbalanced |

---

> **The key takeaway:** A neural network is not magic. It is a structured system of adjustable numbers that gets better through repetition and feedback — just like learning any new skill. The "intelligence" comes entirely from the data it is trained on.
